# MLP: a primeira rede neural

**Para quem está começando.** Nenhuma linha de código aqui assume que você já viu rede neural. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. olhar um problema que nenhuma reta separa;
2. treinar um MLP com PyTorch e ver a fronteira deixar de ser reta;
3. ler a curva de perda do treino;
4. comparar com a regressão logística num dado tabular de verdade.

Roda no Google Colab, que já traz o PyTorch. No Jupyter do seu computador, precisa de `torch`, `numpy`, `pandas`, `scikit-learn` e `matplotlib`; o ambiente descrito no `README.md` já tem tudo, e a versão usada aqui é `torch==2.13.0`.


## O que é um MLP, sem jargão

Um neurônio, na versão de computador, faz duas coisas: multiplica cada entrada por um peso, soma tudo e passa o resultado por uma função que não é reta (a ativação). O `ReLU`, usado aqui, é a mais simples delas: devolve o valor se ele for positivo e zero se for negativo.

Um MLP (perceptron de múltiplas camadas) é um conjunto desses neurônios em camadas, com a saída de uma alimentando a entrada da seguinte. A camada do meio é chamada de escondida, e é ali que está o pulo do gato: ela **aprende colunas novas** a partir das originais. É exatamente o que o polinômio fazia no notebook de regressão, com uma diferença: as colunas novas das redes não são escolhidas por você, são ajustadas pelo treino, e não têm forma fixa.

O treino é uma descida de gradiente como qualquer outra. O que a rede acrescenta é o **backpropagation**, que é a regra da cadeia aplicada camada por camada: ele responde quanto cada peso contribuiu para o erro, de trás para frente, da saída até a entrada. Comparado à regressão logística, o MLP tem a mesma função de perda e o mesmo otimizador; o que ele ganha é a camada escondida e a ativação não linear.


## Passo 0: ambiente e configuração

A semente é a mesma em todas as células, e ela é fixada duas vezes porque as duas bibliotecas sorteiam pesos: `numpy` para os dados e `torch` para a inicialização da rede. Se o `torch` não estiver instalado, `%pip install torch==2.13.0`.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (9, 4)
print(f"✅ torch {torch.__version__} em {device} | seed {RANDOM_STATE}")

## Passo 1: dois círculos entrelaçados

O conjunto `make_moons` desenha duas meias-luas que se entrelaçam. Uma reta não separa as duas classes, então a regressão logística fica presa num teto de acurácia, e não é por falta de atributos: com dois atributos, é o formato da fronteira que a limita.


In [ ]:
X_moon, y_moon = make_moons(n_samples=1000, noise=0.2, random_state=RANDOM_STATE)
X_moon = StandardScaler().fit_transform(X_moon)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap="coolwarm", edgecolor="k", s=15)
ax.set_title("Duas meias-luas: nenhuma reta separa")
plt.show()

logistica = LogisticRegression().fit(X_moon, y_moon)
print(f"📊 regressão logística: acurácia = {logistica.score(X_moon, y_moon):.4f}")

## Passo 2: uma camada escondida muda a fronteira

A rede abaixo tem 2 entradas, duas camadas escondidas de 16 neurônios e 1 saída. A saída é um número, e a função de perda `BCEWithLogitsLoss` transforma esse número em probabilidade e mede o erro, o mesmo papel que a logística cumpria no modelo linear.

O treino é em lote completo, ou seja, a rede vê os 1000 exemplos em cada passo (com 1000 linhas, o lote completo é rápido e simplifica o código do loop). O histórico guarda a perda e a acurácia de cada época, que é o que vamos ler no gráfico.


In [ ]:
class MLP(nn.Module):
    """Two hidden layers with ReLU and a single logit output."""

    def __init__(self, n_features: int, hidden: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def to_tensor(X, y=None):
    """Numpy in, float32 tensor out: PyTorch works with its own arrays."""
    X_t = torch.tensor(np.asarray(X, dtype=np.float32))
    if y is None:
        return X_t
    return X_t, torch.tensor(np.asarray(y, dtype=np.float32)).view(-1, 1)


def predict_proba(model: nn.Module, X) -> np.ndarray:
    """Probability of class 1, as a plain numpy array."""
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(to_tensor(X))).numpy().ravel()


def train_model(model: nn.Module, X, y, epochs: int = 300, lr: float = 0.05):
    """Full-batch training; returns (loss, accuracy) per epoch."""
    X_t, y_t = to_tensor(X, y)
    otimizador = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    historico = []
    for _ in range(epochs):
        model.train()
        otimizador.zero_grad()
        loss = loss_fn(model(X_t), y_t)
        loss.backward()
        otimizador.step()
        acuracia = float(((predict_proba(model, X) > 0.5) == y).mean())
        historico.append((float(loss.item()), acuracia))
    return historico

In [ ]:
rede_moon = MLP(n_features=2)
inicio = time.perf_counter()
historico = train_model(rede_moon, X_moon, y_moon, epochs=300, lr=0.05)
print(f"⏱️ 300 épocas em {time.perf_counter() - inicio:.1f}s")

historico = np.array(historico)
fig, ax = plt.subplots()
ax.plot(historico[:, 0], color="#0F6B68")
ax.set_xlabel("época")
ax.set_ylabel("perda (BCE)")
ax.set_title("Curva de perda do treino")
plt.show()

acuracia_rede = float(((predict_proba(rede_moon, X_moon) > 0.5) == y_moon).mean())
print(f"📊 MLP: acurácia = {acuracia_rede:.4f} | regressão logística = {logistica.score(X_moon, y_moon):.4f}")

### Como ler isso

A perda cai rápido nas primeiras dezenas de épocas e depois quase não muda. Esse é o formato normal: o que a rede aprende primeiro é a forma geral da fronteira, e o ajuste fino rende pouco.

Três coisas para experimentar, todas de uma linha:

- suba as épocas para 3 000 e compare as duas acurácias. Se a acurácia de treino chegar muito perto de 1,0 com a perda quase zero, a rede passou a decorar o ruído das meias-luas;
- troque `hidden=16` por `hidden=2`. A rede fica quase linear, e a fronteira volta perto da reta;
- remova o `ReLU`. Sem ativação não linear, empilhar camadas equivale a uma única transformação linear, por mais neurônios que você ponha. A acurácia volta ao teto da regressão logística, e isso não é coincidência.

A acurácia que o `print` mostra é sobre o próprio treino, então ela é otimista. No próximo passo a mesma rede é treinada num dado separado em treino e teste, onde o número é honesto.


In [ ]:
xx, yy = np.meshgrid(np.linspace(X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5, 200),
                     np.linspace(X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5, 200))
grade = np.c_[xx.ravel(), yy.ravel()]

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for eixo, (titulo, probabilidade) in zip(eixos, [
    ("Regressão logística", logistica.predict_proba(grade)[:, 1]),
    ("MLP (2-16-16-1)", predict_proba(rede_moon, grade)),
]):
    eixo.contourf(xx, yy, probabilidade.reshape(xx.shape), levels=10, alpha=0.3, cmap="coolwarm")
    eixo.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap="coolwarm", edgecolor="k", s=12)
    eixo.set_title(titulo)
plt.tight_layout()
plt.show()

## Passo 3: e num dado tabular de verdade?

As meias-luas são o caso fácil, feito para o desenho ficar visível. A pergunta que importa é outra: num dado tabular comum, com atributos que já vêm em colunas, a rede ganha do modelo linear?

O teste é honesto: mesma divisão de treino e teste, padronização dentro do `Pipeline`, e a rede avaliada em **três sementes**, porque o resultado dela muda com a inicialização dos pesos. A regressão logística tem uma resposta só, mas a rede precisa de média e desvio.


In [ ]:
dados = load_breast_cancer()
X_treino, X_teste, y_treino, y_teste = train_test_split(
    dados.data, dados.target, test_size=0.25, stratify=dados.target, random_state=RANDOM_STATE,
)
escala = StandardScaler().fit(X_treino)
X_treino_pad, X_teste_pad = escala.transform(X_treino), escala.transform(X_teste)

inicio = time.perf_counter()
modelo_linear = LogisticRegression(max_iter=5000).fit(X_treino_pad, y_treino)
tempo_linear = time.perf_counter() - inicio

acuracias, tempo_rede = [], 0.0
for semente in range(3):
    torch.manual_seed(semente)
    rede = MLP(n_features=X_treino_pad.shape[1], hidden=32)
    inicio = time.perf_counter()
    train_model(rede, X_treino_pad, y_treino, epochs=300, lr=0.01)
    tempo_rede += time.perf_counter() - inicio
    acuracias.append(float(((predict_proba(rede, X_teste_pad) > 0.5) == y_teste).mean()))

tabela = pd.DataFrame([
    {"modelo": "Regressão logística", "acuracia_teste": modelo_linear.score(X_teste_pad, y_teste),
     "desvio entre sementes": 0.0, "tempo_s": tempo_linear},
    {"modelo": "MLP (3 sementes)", "acuracia_teste": float(np.mean(acuracias)),
     "desvio entre sementes": float(np.std(acuracias)), "tempo_s": tempo_rede / 3},
]).set_index("modelo")
display(tabela.round(4))

### Como ler isso

Na maioria dos dados tabulares pequenos, a rede não ganha do modelo linear, e este notebook é um exemplo disso. Não é defeito da rede: é que os atributos já vêm prontos em colunas, e a fronteira útil costuma ser simples. Onde a rede se paga é em dado não estruturado (imagem, texto, áudio), onde não existe coluna pronta e a própria representação precisa ser aprendida.

O preço da rede aparece nas colunas da tabela:

- **`desvio entre sementes`**: com a mesma receita e sementes diferentes, a acurácia muda. Qualquer comparação publicada com rede neural precisa disso, e a do modelo linear é sempre a mesma;
- **`tempo_s`**: a rede treina mais devagar e tem botões que o modelo linear não tem (camadas, época, taxa de aprendizado). Cada botão é uma decisão que alguém precisa justificar;
- **padronização**: como no kNN e no SVM, a rede depende da escala das entradas, e aqui o `StandardScaler` está fora do laço de treino de propósito, para os três treinos verem exatamente a mesma tabela.

## Resumo

Um MLP é uma regressão logística com uma camada escondida e uma ativação não linear no meio. Essa camada aprende colunas novas em vez de receber as suas, e é isso que permite a fronteira deixar de ser reta; sem a ativação, empilhar camadas não muda nada. O treino é descida de gradiente com backpropagation, e o resto do vocabulário (perda, época, lote, taxa de aprendizado) é o mesmo de qualquer modelo ajustado por gradiente.

A lição prática: rede neural não é upgrade automático. Em tabela pequena, o modelo simples costuma empatar ou ganhar, com uma fração do tempo e nenhum botão para ajustar. Guarde a rede para quando a representação precisar ser aprendida.

**Próximo passo:** trocar as camadas densas por convoluções, que é o mesmo mecanismo com uma restrição a mais (cada neurônio olha apenas uma vizinhança da entrada), no caso de imagem.

### Referências

**Backpropagation e o MLP**

Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). *Learning representations by back-propagating errors.* Nature 323, 533-536.

Hornik, K., Stinchcombe, M., & White, H. (1989). *Multilayer feedforward networks are universal approximators.* Neural Networks 2(5), 359-366.

**O contexto**

Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning.* MIT Press. Capítulo 6, sobre redes densas e backpropagation.

```bibtex
@article{rumelhart1986learning,
  title   = {Learning representations by back-propagating errors},
  author  = {Rumelhart, David E. and Hinton, Geoffrey E. and Williams, Ronald J.},
  journal = {Nature},
  volume  = {323},
  pages   = {533--536},
  year    = {1986}
}

@article{hornik1989multilayer,
  title   = {Multilayer feedforward networks are universal approximators},
  author  = {Hornik, Kurt and Stinchcombe, Maxwell and White, Halbert},
  journal = {Neural Networks},
  volume  = {2},
  number  = {5},
  pages   = {359--366},
  year    = {1989}
}

@book{goodfellow2016deep,
  title     = {Deep Learning},
  author    = {Goodfellow, Ian and Bengio, Yoshua and Courville, Aaron},
  year      = {2016},
  publisher = {MIT Press}
}
```
